# BDA — Analitik Pendukung Model Lonjakan Respiratory (PySpark)
**File:** `bda_2_09020624025.ipynb` — **VERSI 2 (REPOSISI setelah seleksi 1 model untuk 1 fitur web)**

## CHANGELOG vs versi sebelumnya
| # | Versi 1 (sebelumnya) | Versi 2 (sekarang) |
|---|---|---|
| Peran notebook | Model regresi time series TERPISAH (GBTRegressor, prediksi angka admisi minggu depan) | **Notebook analitik pendukung** untuk model tunggal di `model_2` (tidak melatih model baru) |
| Isi | Train/tune/evaluate/simpan model sendiri | Analisis tren musiman, perbandingan antar-region, validasi/interpretasi `models/surge_classifier_rf` |
| Alasan perubahan | - | Setelah diseleksi jadi 1 model utk 1 fitur web, dua model terpisah (klasifikasi gejala + regresi time series) jadi berlebihan. `model_2` sekarang satu-satunya model produksi; `bda_2` diposisikan ulang sbg pendukung "Big Data Analytics" — insight tambahan dari data besar (panel 51 region) yang berguna utk dashboard RS, tanpa duplikasi model. |

**Isi notebook ini:**
1. Tren musiman nasional (kapan biasanya lonjakan terjadi dalam setahun)
2. Perbandingan pola antar-region (big-data angle: agregasi lintas 51 region × 220+ minggu)
3. Load model dari `model_2` dan tampilkan feature importance + evaluasi tambahan (precision/recall per kelas)
4. Insight yang bisa dipakai utk narasi dashboard RS Surabaya


## 0. Instalasi Library
Jalankan sekali di awal. Notebook ini hanya baca model yang sudah disimpan oleh `model_2`, jadi tidak perlu kagglehub — cukup PySpark, pandas, dan matplotlib.

In [ ]:
!pip install -q pyspark==3.5.1 pandas matplotlib

ERROR: Operation cancelled by user


## 1. Import Library

In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.classification import RandomForestClassificationModel
from pyspark.ml.feature import StringIndexerModel, VectorAssembler
from pyspark.ml.evaluation import MulticlassClassificationEvaluator


## 2. Spark Session

In [ ]:
spark = (
    SparkSession.builder
    .appName("RespiratorySurge_BigDataAnalytics")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.ansi.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark


## 3. Load Data + Rekonstruksi Fitur (sama persis seperti `model_2`)

In [ ]:
PROCESSED_DIR = "../data/processed"
MODELS_DIR = "../models"

df_raw = spark.read.parquet(os.path.join(PROCESSED_DIR, "hospital_time_series.parquet"))

HOSP_MAP = {
    "Week Ending Date": "week_date", "Geographic aggregation": "state",
    "Total Patients Hospitalized with COVID-19": "total_covid_hosp",
    "Total Patients Hospitalized with Influenza": "total_flu_hosp",
    "Total Patients Hospitalized with RSV": "total_rsv_hosp",
    "Number of Inpatient Beds Occupied": "inpatient_beds_occupied",
    "Number of Inpatient Beds": "inpatient_beds_total",
}
df = df_raw
for old, new in HOSP_MAP.items():
    df = df.withColumnRenamed(old, new)
df = df.withColumn("week_date", F.to_date("week_date"))

EXCLUDE_REGIONS = ["USA", "AS", "GU", "MP", "VI", "PR"]
df = df.filter(~F.col("state").isin(EXCLUDE_REGIONS))
for c in ["total_covid_hosp", "total_flu_hosp", "total_rsv_hosp"]:
    df = df.fillna({c: 0})
df = df.withColumn("total_resp", F.col("total_covid_hosp") + F.col("total_flu_hosp") + F.col("total_rsv_hosp"))
df = df.withColumn("bed_occ_ratio",
                    F.when(F.col("inpatient_beds_total") > 0, F.col("inpatient_beds_occupied") / F.col("inpatient_beds_total")))

w = Window.partitionBy("state").orderBy("week_date")
df = (df
    .withColumn("lag_1", F.lag("total_resp", 1).over(w))
    .withColumn("lag_2", F.lag("total_resp", 2).over(w))
    .withColumn("rolling_mean_4w", F.avg("total_resp").over(w.rowsBetween(-4, -1)))
    .withColumn("bed_occ_ratio_lag1", F.lag("bed_occ_ratio", 1).over(w))
    .withColumn("week_of_year", F.weekofyear("week_date"))
    .withColumn("month", F.month("week_date"))
    .withColumn("pct_change", F.when(F.col("lag_1") > 0, (F.col("total_resp") - F.col("lag_1")) / F.col("lag_1")))
)
df_clean = df.dropna(subset=["lag_1", "lag_2", "rolling_mean_4w", "bed_occ_ratio_lag1", "pct_change"])

SPIKE_THRESHOLD = df_clean.approxQuantile("pct_change", [0.70], 0.01)[0]
df_clean = df_clean.withColumn("spike_label", (F.col("pct_change") > F.lit(SPIKE_THRESHOLD)).cast("int"))
print("Jumlah baris:", df_clean.count(), "| Threshold lonjakan:", f"{SPIKE_THRESHOLD:.1%}")


## 4. Insight #1 — Tren Musiman Nasional
Kapan dalam setahun lonjakan paling sering terjadi? Berguna utk kesiapan RS secara musiman (mis. musim hujan/akhir tahun).

In [ ]:
monthly_spike_rate = (df_clean.groupBy("month")
                       .agg(F.avg("spike_label").alias("spike_rate"), F.count("*").alias("n_obs"))
                       .orderBy("month").toPandas())
monthly_spike_rate["month_name"] = pd.to_datetime(monthly_spike_rate["month"], format="%m").dt.strftime("%b")

plt.figure(figsize=(8,4))
plt.bar(monthly_spike_rate["month_name"], monthly_spike_rate["spike_rate"]*100, color="#4C72B0")
plt.axhline(30, color="red", linestyle="--", linewidth=1, label="Rata-rata keseluruhan (~30%)")
plt.title("Persentase Minggu dengan Lonjakan per Bulan (seluruh region, 2020-2024)")
plt.ylabel("% Minggu Lonjakan")
plt.legend()
plt.tight_layout()
plt.show()

print(monthly_spike_rate[["month_name","spike_rate","n_obs"]])


## 5. Insight #2 — Perbandingan Antar-Region (Big Data angle)
Agregasi lintas 51 region menunjukkan variasi pola lonjakan — relevan utk menjelaskan kenapa model memakai `state` sbg fitur.

In [ ]:
region_spike_rate = (df_clean.groupBy("state")
                      .agg(F.avg("spike_label").alias("spike_rate"), F.avg("total_resp").alias("avg_admisi"))
                      .orderBy(F.desc("spike_rate")).toPandas())

print("10 region dengan tingkat lonjakan tertinggi:")
print(region_spike_rate.head(10).to_string(index=False))
print("\n10 region dengan tingkat lonjakan terendah:")
print(region_spike_rate.tail(10).to_string(index=False))


In [ ]:
plt.figure(figsize=(10,4))
top15 = region_spike_rate.head(15)
plt.bar(top15["state"], top15["spike_rate"]*100, color="#C44E52")
plt.title("15 Region dengan Tingkat Lonjakan Tertinggi (% minggu lonjakan)")
plt.ylabel("% Minggu Lonjakan")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 6. Load Model dari `model_2_09020624025.ipynb` + Feature Importance

In [ ]:
loaded_model = RandomForestClassificationModel.load(os.path.join(MODELS_DIR, "surge_classifier_rf"))
loaded_state_indexer = StringIndexerModel.load(os.path.join(MODELS_DIR, "state_indexer"))

feature_cols = ["lag_1", "lag_2", "rolling_mean_4w", "bed_occ_ratio_lag1", "week_of_year", "state_idx"]
importances = list(zip(feature_cols, loaded_model.featureImportances.toArray()))
importances.sort(key=lambda x: -x[1])
print("Feature importance (dari model tersimpan):")
for f, imp in importances:
    print(f"  {f:20s}: {imp:.4f}")


## 7. Evaluasi Tambahan: Precision & Recall per Kelas
Confusion Matrix sudah ditampilkan di `model_2`. Di sini ditambahkan precision/recall per kelas — penting utk konteks RS: *recall* kelas "Lonjakan" paling krusial (model tidak boleh terlalu sering melewatkan lonjakan nyata).

In [ ]:
df_idx = loaded_state_indexer.transform(df_clean)
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_feat = assembler.transform(df_idx)

_, _, test_df = df_feat.randomSplit([0.7, 0.2, 0.1], seed=42)
test_pred = loaded_model.transform(test_df)

precision_eval = MulticlassClassificationEvaluator(labelCol="spike_label", predictionCol="prediction",
                                                     metricName="precisionByLabel", metricLabel=1.0)
recall_eval = MulticlassClassificationEvaluator(labelCol="spike_label", predictionCol="prediction",
                                                 metricName="recallByLabel", metricLabel=1.0)

precision_spike = precision_eval.evaluate(test_pred)
recall_spike = recall_eval.evaluate(test_pred)
print(f"Precision (kelas 'Lonjakan'): {precision_spike:.2%}")
print(f"Recall (kelas 'Lonjakan')   : {recall_spike:.2%}")
print("\nArtinya: dari semua lonjakan yang benar-benar terjadi di test set, model menangkap "
      f"{recall_spike:.0%} di antaranya (recall) — metrik ini lebih penting dari akurasi keseluruhan "
      "utk use-case early-warning RS, karena melewatkan lonjakan nyata lebih berisiko drpd false alarm.")


## 8. Ringkasan untuk Dashboard RS Surabaya
- **Pola musiman**: lonjakan cenderung lebih sering terjadi di bulan-bulan tertentu (lihat grafik Insight #1) — bisa jadi dasar kesiapan RS secara musiman.
- **Variasi antar-region**: tingkat lonjakan berbeda signifikan antar wilayah (Insight #2) — mendukung keputusan desain `model_2` yang memasukkan `state`/region sbg fitur, bukan model generik satu-ukuran-utk-semua.
- **Fitur paling berpengaruh**: tren terkini (`lag_1`, `rolling_mean_4w`) + identitas region adalah prediktor utama — konsisten dgn intuisi domain (lonjakan biasanya didahului tren naik bertahap).
- **Precision/Recall kelas "Lonjakan"**: dilaporkan terpisah dari akurasi keseluruhan karena lebih relevan utk keputusan operasional (kapan harus siap-siap nambah kapasitas).
- Catatan yang sama dgn `model_2`: ini data proxy AS, bukan data riil RS Surabaya — pola musiman & regional di atas perlu divalidasi ulang begitu ada data riil.
